In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


df = pd.read_csv("/content/Online Retail.csv", encoding='latin1')


print("Original dataset shape:", df.shape)


duplicate_records = df.duplicated().sum()


print("Duplicate records:", duplicate_records)


df = df.drop_duplicates().copy()


duplicate_invoice_numbers = df[df.duplicated("InvoiceNo", keep=False)]["InvoiceNo"].nunique()
print("Invoice numbers appearing on multiple rows:",
      duplicate_invoice_numbers)


df["CustomerID"] = df["CustomerID"].fillna("Unknown").astype(str)


df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)


df = df.dropna(subset=["InvoiceDate"]).copy()


summary_before = df[["Quantity", "UnitPrice"]].describe()


print("\nSummary Statistics BEFORE preprocessing:")
print(summary_before)


country_encoded = pd.get_dummies(
    df["Country"],
    prefix="Country",
    dtype=int
)


df = pd.concat(
    [df.drop(columns=["Country"]), country_encoded],
    axis=1
)




def find_iqr_bounds(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)


    IQR = Q3 - Q1


    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR


    return lower_bound, upper_bound




quantity_lower, quantity_upper = find_iqr_bounds(
    df["Quantity"]
)


quantity_outliers = (
    (df["Quantity"] < quantity_lower) |
    (df["Quantity"] > quantity_upper)
).sum()


price_lower, price_upper = find_iqr_bounds(
    df["UnitPrice"]
)


unitprice_outliers = (
    (df["UnitPrice"] < price_lower) |
    (df["UnitPrice"] > price_upper)
).sum()


print("\nQuantity outliers:", quantity_outliers)
print("UnitPrice outliers:", unitprice_outliers)


df = df[
    df["Quantity"].between(
        quantity_lower,
        quantity_upper
    )
    &
    df["UnitPrice"].between(
        price_lower,
        price_upper
    )
].copy()


scaler = MinMaxScaler()


df[["Quantity", "UnitPrice"]] = scaler.fit_transform(
    df[["Quantity", "UnitPrice"]]
)


summary_after = df[["Quantity", "UnitPrice"]].describe()


print("\nSummary Statistics AFTER preprocessing:")
print(summary_after)


print("\nFinal dataset shape:", df.shape)


df.to_csv(
    "retail_sales_cleaned.csv",
    index=False
)


print("\nCleaned dataset saved successfully!")


print("URK25CS7102")
